# 🚀 Inspección Vogon de Datos — Live Coding
### Masterclass: Desarrollo robusto y validación estricta de tipos con Pydantic

**Bloque 2 — Live coding / Demo de código (25 min)**

Bienvenidos al Servicio de Aduanas Intergaláctico. Hoy sois Inspectores Vogon: nadie sube a la nave si su formulario no está impecable. Vamos a construir ese sistema de inspección automática con **Pydantic**.


## 0. El problema, sin Pydantic

Antes de nada, veamos por qué Python "normal" no nos protege, aunque usemos *type hints*.


In [1]:
def registrar_pasajero(nombre: str, edad: int):
    print(f"Pasajero {nombre}, edad {edad}, sube a la nave dentro de {edad} años luz")
    return edad * 2

# Python NO valida los type hints en tiempo de ejecución.
# Esto "funciona" aunque no tenga ningún sentido:
resultado = registrar_pasajero("Zaphod", "no soy un número")
print(resultado)


Pasajero Zaphod, edad no soy un número, sube a la nave dentro de no soy un número años luz
no soy un númerono soy un número


👆 Esto debería explotar aquí, pero Python nos deja pasar el error a otra parte del código (o a producción). **Ese es exactamente el problema que resuelve Pydantic.**


## 1. Instalación e importación

In [2]:
# En vuestro entorno local:
# pip install pydantic

from pydantic import BaseModel, Field, EmailStr, field_validator, ValidationError
from typing import Literal, Optional


## 2. Nuestro primer modelo: `PasajeroGalactico`

Un modelo Pydantic es una clase que hereda de `BaseModel`. Cada atributo con su *type hint* se convierte en una regla de validación.


In [3]:
class PasajeroGalactico(BaseModel):
    nombre: str
    edad: int

# Caso válido
p1 = PasajeroGalactico(nombre="Arthur Dent", edad=30)
print(p1)


nombre='Arthur Dent' edad=30


In [4]:
# Caso con coerción automática: "30" (string) se convierte a 30 (int)
p2 = PasajeroGalactico(nombre="Ford Prefect", edad="42")
print(p2)
print(type(p2.edad))


nombre='Ford Prefect' edad=42
<class 'int'>


In [5]:
# Caso inválido: esto SÍ debe fallar, y fallar con un mensaje claro
try:
    p3 = PasajeroGalactico(nombre="Trillian", edad="no soy un número")
except ValidationError as e:
    print(e)


1 validation error for PasajeroGalactico
edad
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='no soy un número', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


**Puntos clave para el debate:**
- Pydantic intenta *convertir* tipos compatibles (`"42"` → `42`), no solo rechazar.
- Si no puede convertir, lanza `ValidationError` con el campo exacto que falló.
- Esto pasa **al crear el objeto**, no tres funciones más tarde.


## 3. Restringiendo valores con `Field()`

No basta con "es un int". Queremos: la edad debe ser positiva, el nombre no puede estar vacío.


In [6]:
class PasajeroGalactico(BaseModel):
    nombre: str = Field(..., min_length=1, max_length=50, description="Nombre del pasajero")
    edad: int = Field(..., gt=0, le=1000, description="Edad en años terrestres")
    especie: Literal["humano", "vogon", "betelgeusiano", "androide"]
    planeta_origen: str


Probemos a crear un pasajero con una edad absurda (`edad=50000000000`). Debería fallar porque supera el límite (`le=1000`):


In [7]:
try:
    p4 = PasajeroGalactico(nombre="Marvin", edad=50000000000, especie="androide", planeta_origen="Magrathea")
except ValidationError as e:
    print(e)


1 validation error for PasajeroGalactico
edad
  Input should be less than or equal to 1000 [type=less_than_equal, input_value=50000000000, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


**`Literal`** es interesante: solo acepta esos valores exactos, como un `enum` ligero. Probad a poner `especie="klingon"` y ved el error.


## 4. Validación de formato: `EmailStr`

Pydantic trae tipos "especiales" ya validados para casos comunes (email, URL, etc.). Requiere el extra `email-validator` (`pip install pydantic[email]`).


In [8]:
class PasajeroGalactico(BaseModel):
    nombre: str = Field(..., min_length=1, max_length=50)
    edad: int = Field(..., gt=0, le=1000)
    especie: Literal["humano", "vogon", "betelgeusiano", "androide"]
    planeta_origen: str
    contacto: EmailStr

try:
    p5 = PasajeroGalactico(
        nombre="Zaphod Beeblebrox", edad=200, especie="betelgeusiano",
        planeta_origen="Betelgeuse V", contacto="no-es-un-email"
    )
except ValidationError as e:
    print(e)


1 validation error for PasajeroGalactico
contacto
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='no-es-un-email', input_type=str]


## 5. Validadores personalizados con `@field_validator`

¿Y si la regla no es un tipo estándar? Ej: el nombre no puede ser solo espacios en blanco.


In [9]:
class PasajeroGalactico(BaseModel):
    nombre: str = Field(..., min_length=1, max_length=50)
    edad: int = Field(..., gt=0, le=1000)
    especie: Literal["humano", "vogon", "betelgeusiano", "androide"]
    planeta_origen: str
    contacto: EmailStr

    @field_validator("nombre")
    @classmethod
    def nombre_no_vacio(cls, valor: str) -> str:
        if valor.strip() == "":
            raise ValueError("el nombre no puede estar vacío ni ser solo espacios")
        return valor.strip().title()  # normalizamos también

try:
    p6 = PasajeroGalactico(
        nombre="   ", edad=30, especie="humano",
        planeta_origen="Tierra", contacto="arthur@tierra.gal"
    )
except ValidationError as e:
    print(e)


1 validation error for PasajeroGalactico
nombre
  Value error, el nombre no puede estar vacío ni ser solo espacios [type=value_error, input_value='   ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


## 6. Modelos anidados: `Equipaje` dentro de `PasajeroGalactico`

Un `BaseModel` puede tener como campo... otro `BaseModel`. Esto es MUY común en la vida real (ej. una `Direccion` dentro de un `Usuario`, o `metadata` dentro de un registro de datos para ML).


In [10]:
class Equipaje(BaseModel):
    peso_kg: float = Field(..., gt=0, le=50)
    contiene_toalla: bool = True  # todo buen autoestopista lleva una toalla

class PasajeroGalactico(BaseModel):
    nombre: str = Field(..., min_length=1, max_length=50)
    edad: int = Field(..., gt=0, le=1000)
    especie: Literal["humano", "vogon", "betelgeusiano", "androide"]
    planeta_origen: str
    contacto: EmailStr
    equipaje: Equipaje

    @field_validator("nombre")
    @classmethod
    def nombre_no_vacio(cls, valor: str) -> str:
        if valor.strip() == "":
            raise ValueError("el nombre no puede estar vacío ni ser solo espacios")
        return valor.strip().title()

# Pydantic valida el diccionario anidado automáticamente
p7 = PasajeroGalactico(
    nombre="arthur dent", edad=30, especie="humano", planeta_origen="Tierra",
    contacto="arthur@tierra.gal",
    equipaje={"peso_kg": 12.5, "contiene_toalla": True}
)
print(p7)
print(p7.equipaje.contiene_toalla)


nombre='Arthur Dent' edad=30 especie='humano' planeta_origen='Tierra' contacto='arthur@tierra.gal' equipaje=Equipaje(peso_kg=12.5, contiene_toalla=True)
True


## 7. Debate rápido (2-3 min)

- ¿Dónde meteríais esto en un proyecto real de ML? (pista: validar un CSV/JSON antes de dárselo al modelo)
- ¿Qué pasa si tenéis 10.000 registros y uno solo falla la validación? ¿Debería parar todo el pipeline o solo descartar ese registro?
- ¿Puede la validación estricta *excluir* datos legítimos por error (ej. nombres con formatos poco comunes, teléfonos internacionales)?

👉 Ahora pasamos al reto: vais a construir vuestro propio modelo Pydantic para un caso nuevo.
